# Tipado y modelos de datos con Pydantic

**Unidad 1 · Sesión 2**

Anotaciones de tipo, análisis estático y validación de datos en Python.

## Contenido

| Sección | Tema |
|---|---|
| 0 | Preparación |
| 1 | Anotaciones de tipo y colecciones |
| 2 | Análisis estático con mypy |
| Ejemplo | Descuentos y precisión del tipo de retorno |
| 3 | Modelos con Pydantic |
| 4 | Restricciones y errores de validación |
| 5 | Validadores de campos |
| 6 | Configuración |
| Ejemplo | Registro de usuarios |
| 7 | Modelos anidados y serialización |
| 8 | Módulo reutilizable |
| 9 | Procesamiento de registros |

## Objetivos

- Anotar funciones y colecciones e interpretar diagnósticos de mypy.
- Definir modelos con restricciones y validadores.
- Localizar errores en datos de entrada.
- Serializar modelos anidados y utilizarlos desde un módulo.

## 0. Preparación

>**mypy** es un verificador estático: analiza el código y contrasta las operaciones
con sus tipos declarados. **Pydantic** es una biblioteca que utiliza anotaciones
para validar y convertir datos durante la ejecución.

Ejecuta la siguiente celda en Colab o Jupyter. Desde terminal puedes instalar las
mismas dependencias con `python -m pip install -r requirements.txt`.

In [ ]:
%pip install -q pydantic==2.12.5 mypy==1.19.1 email-validator==2.3.0

In [ ]:
import sys
from importlib.metadata import version

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer"
print(f"Python: {sys.version.split()[0]}")
print(f"Pydantic: {version('pydantic')}")
print(f"mypy: {version('mypy')}")

## 1. Del comportamiento al contrato

Una anotación indica el tipo esperado. En `score: float`, `score` es el nombre y
`float` el tipo. Después de los parámetros, `-> str` describe el valor de retorno.
La implementación sigue siendo necesaria: los tipos no calculan el resultado.

**Las anotaciones no validan ni convierten automáticamente los argumentos cuando
Python ejecuta una función.**

In [ ]:
def repeat_text(text: str, times: int) -> str:
    return text * times


print(repeat_text("hello ", 2))
# Intentional mismatch: Python executes this even though the annotation says str.
print(repeat_text(3, 2))
assert repeat_text(3, 2) == 6

### Análisis estático y validación durante la ejecución

Las anotaciones describen los tipos esperados de variables, parámetros y retornos.
Mypy utiliza esa información para detectar operaciones incompatibles sin ejecutar
el programa. Pydantic utiliza los campos de un modelo para validar datos al crear
una instancia. Las comprobaciones con `assert` verifican resultados concretos.

Estas herramientas tienen alcances diferentes. Una función puede superar el
análisis estático y recibir después un dato externo inválido.

### Colecciones y valores que pueden faltar

`list[float]` describe una lista de números; `dict[str, int]`, un diccionario de
claves de texto y valores enteros. `float | None` permite un número o ausencia.
No significa «cero». Un argumento con `= None` además puede omitirse en la llamada.

In [ ]:
scores: list[float] = [0.0, 0.5, 1.0]
counts: dict[str, int] = {"positive": 2, "neutral": 1}
coordinates: tuple[float, float] = (20.97, -89.62)


def mean_score(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


assert mean_score([]) is None
assert mean_score([0.0]) == 0.0
print(mean_score(scores))

### Ejercicio 1 · Anotar sin cambiar el comportamiento

Anota `select_scores`: recibe una lista de floats y un umbral float; devuelve
una lista de floats. Conserva los valores iguales al umbral. Comprueba lista vacía,
umbral cero y `[0.5, 0.9, 1.0]` con umbral 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Add annotations and checks.
def select_scores(values, threshold):
    return [value for value in values if value >= threshold]

## 2. Leer un error del verificador

**mypy** analiza las anotaciones y detecta usos incompatibles de los tipos. Esta celda
analiza una cadena de código sin ejecutarla ni crear archivos. `api.run` devuelve
salida, errores internos y código de salida. El código 1 es esperado aquí: hemos
introducido dos errores deliberados.

Lee el mensaje en este orden: ubicación, operación, tipo recibido y tipo esperado.
No cambies un tipo para callar al verificador: revisa qué comportamiento necesitas.

In [ ]:
from mypy import api as mypy_api

broken_source = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold("0.9", 0.8)

def display_score(score: float | None) -> str:
    return f"{score * 100:.1f}%"
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", broken_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout
assert "operator" in stdout

### Ejercicio 2 · Corregir el contrato

Corrige los dos problemas de `broken_source`: convierte explícitamente el texto
a float y atiende `None` antes de multiplicar. Trabaja en otra cadena para conservar
el ejemplo defectuoso. Ejecuta mypy y comprueba código de salida 0. ¿Por qué convertir
con `float` todavía puede fallar en tiempo de ejecución?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Write fixed_source and run mypy_api.run on it.

### Precisión del tipo de retorno

El tipo de retorno determina qué operaciones puede comprobar el verificador.
`Iterable[float]` garantiza que los valores pueden recorrerse, pero no garantiza
`len()` ni acceso por índice.

La función siguiente devuelve una lista y declara `list[float]` como retorno.
`Iterable` describe la entrada sin exigir que sea una lista. Su funcionamiento
se estudiará con detalle en la próxima sesión.

In [ ]:
from collections.abc import Iterable


def calculate_discounts(items: Iterable[float], discount: float) -> list[float]:
    return [item * (1 - discount) for item in items]


prices = [100.0, 200.0, 300.0]
discounted_prices = calculate_discounts(prices, 0.2)
print(len(discounted_prices))
print(discounted_prices[0])
assert discounted_prices == [80.0, 160.0, 240.0]

Cambia solo el retorno a `Iterable[float]` en una copia del ejemplo y analízala
con mypy. Compara los diagnósticos con el resultado de ejecutar Python.

### Evitar `Any` como respuesta automática

`Any` permite saltarse muchas comprobaciones. Si un dato crudo puede ser cualquier
objeto, `object` obliga a comprobar su tipo antes de usar operaciones específicas.
Con `isinstance` podemos acotar el tipo. En los modelos, `Literal` limitará los
valores permitidos. Hoy no necesitamos genéricos avanzados ni protocolos.

In [ ]:
def describe_input(value: object) -> str:
    if isinstance(value, str):
        return f"Text with {len(value)} characters"
    return "Not a text value"


print(describe_input("hello"))
print(describe_input(None))

## 3. Un modelo de datos también es una clase

`BaseModel` es la clase base de Pydantic. Al heredar de ella y declarar campos,
obtenemos construcción, validación y serialización. Una instancia sigue teniendo
atributos: `prediction.score`, en lugar de `record["score"]`.

Comenzamos con un modelo deliberadamente pequeño para observar sus límites.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


class BasicPrediction(BaseModel):
    text: str
    score: float


basic_prediction = BasicPrediction.model_validate({"text": "Good service", "score": "0.9"})
print(basic_prediction)
print(type(basic_prediction.score))
assert basic_prediction.score == 0.9
# A float annotation alone does not constrain the numerical range.
print(BasicPrediction(text="Outside the range", score=1.5))

### Conversión y modo estricto

En modo habitual, Pydantic acepta algunas conversiones, como `"0.9"` a `0.9`.
**Convertir** no es lo mismo que **rechazar todo valor cuyo tipo original difiera**.
El modo estricto restringe conversiones; las restricciones de rango son otra regla.

Decidiremos el contrato según la fuente: un archivo puede contener números como
texto, mientras que una configuración interna puede exigir enteros reales.

In [ ]:
try:
    BasicPrediction.model_validate({"text": "Good service", "score": "0.9"}, strict=True)
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    raise AssertionError("Strict validation should reject the string score")

## 4. Campos restringidos y errores útiles

`Literal` enumera etiquetas válidas; `Field` añade límites. `Annotated` combina
un tipo con metadatos: el alias `Score` se puede reutilizar sin repetir sus reglas.
`source: str | None = None` permite ausencia y también omitir el campo.

`extra="forbid"` rechaza claves desconocidas, como `scroe`; de otro modo un error
de escritura podría pasar inadvertido. Este primer modelo aún no normaliza texto.

In [ ]:
from typing import Annotated, Literal

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class ConstrainedPrediction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None


constrained = ConstrainedPrediction(text="Good", label="positive", score=0.0)
assert constrained.source is None
print(constrained)

In [ ]:
invalid_record = {"text": "", "label": "unknown", "score": 1.2, "scroe": 0.8}
try:
    ConstrainedPrediction.model_validate(invalid_record)
except ValidationError as error:
    for detail in error.errors():
        print(detail["loc"], detail["type"], detail["msg"])
else:
    raise AssertionError("The record should be rejected")

Cada entrada de `errors()` describe un problema: `loc` indica el campo, `type`
su categoría y `msg` una explicación. Un registro puede producir varios errores.
La función que procesa un lote debe conservarlos, sin confundir número de errores
con número de registros rechazados.

### Ejercicio 3 · Ausente, nulo e inválido

Prueba registros que omitan `source`, que usen `source=None`, que omitan `score`
y que usen `score=None`. Predice cuáles se aceptan. Comprueba también límites 0 y 1.
Explica por qué `str | None` sin `= None` permitiría nulo pero seguiría siendo obligatorio.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Build each input and catch only ValidationError for expected failures.

## 5. Validadores: reglas que necesitan código

Queremos aceptar `" POSITIVE "` y rechazar textos de solo espacios. Antes de validar
los campos, quitaremos espacios externos y convertiremos las cadenas a minúsculas.
Es una decisión explícita de este taller, no una limpieza universal para todos los textos.

`@field_validator(..., mode="before")` registra una función que recibe la entrada
cruda. `@classmethod` indica que recibe la clase como `cls`, no una instancia como
`self`. Los decoradores conectan nuestra función con Pydantic; no necesitas crear
un decorador propio. Devuelve el valor transformado o lanza `ValueError`.

Los booleanos tienen un comportamiento numérico en Python; queremos rechazarlos
explícitamente como scores. La validación de rango rechazará también `NaN` e infinito.

In [ ]:
class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value

In [ ]:
normalized = Prediction.model_validate(
    {"text": "  GOOD SERVICE  ", "label": " POSITIVE ", "score": "0.9"}
)
assert normalized.text == "good service"
assert normalized.label == "positive"
print(normalized)

### Ejercicio 4 · Detectar los casos que faltaban

Prueba texto `"   "`, etiqueta `None`, score `True`, `"nan"`, `-0.1` y `1.1`.
Deben rechazarse. Comprueba también que `0.0` sea válido y que la entrada original
no se modifique al normalizarla. ¿Por qué `min_length=1` sin quitar espacios era insuficiente?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Validate one changed field at a time and inspect the location of each error.

### Validar asignaciones tiene un alcance concreto

`validate_assignment=True` vuelve a validar al asignar atributos del modelo.
Sin esa opción, construir una instancia válida no garantiza que una asignación
posterior conserve el contrato. Esta opción no vigila cada modificación interna
de una lista: mantenemos las operaciones del lote en funciones conocidas.

In [ ]:
try:
    normalized.score = 2.0
except ValidationError as error:
    print(error.errors()[0]["msg"])
assert normalized.score == 0.9

## 6. Configuración explícita

La configuración también es dato: nombre del lote, umbral de selección y máximo
de registros. Aquí llega como diccionario local, sin variables de entorno ni secretos.

Distingue `model_config` (comportamiento de Pydantic) de `BatchConfig` (parámetros
de nuestra aplicación). El máximo será un entero estricto entre 1 y 1000: no
aceptaremos `"10"` ni `True`. El umbral comparte el contrato de los scores.

In [ ]:
class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value

In [ ]:
config = BatchConfig.model_validate({"batch_name": " Demo batch ", "threshold": "0.9"})
assert config.batch_name == "Demo batch"
assert config.max_records == 100
print(config)

### Ejercicio 5 · Configuración antes de procesar

Construye una configuración con nombre `"Practice"`, umbral 0 y máximo 3.
Después comprueba que máximo `"3"`, máximo 0 y nombre `"   "` produzcan errores.
Explica por qué un error de configuración debe detener el lote completo, mientras
que un registro inválido puede conservarse como rechazo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Create a valid configuration and try the invalid variants.

### Registro de usuarios

El modelo usa `EmailStr` para validar el formato del correo y `SecretStr` para
ocultar la contraseña en su representación habitual. `exclude=True` la excluye
de la serialización. `SecretStr` no cifra ni almacena de forma segura una contraseña.

El rol tiene un valor predeterminado válido. El ejemplo muestra la construcción
del modelo, sus errores de validación y su serialización.

In [ ]:
from pydantic import EmailStr, SecretStr


class User(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=2)
    email: EmailStr
    password: SecretStr = Field(exclude=True)
    role: Literal["author", "editor", "admin"] = "author"


user_data = {
    "name": "Alex",
    "email": "alex@example.com",
    "password": "ExamplePassword123",
}
user = User.model_validate(user_data)
print(user)
print(user.model_dump_json(indent=2))
assert user.role == "author"
assert "password" not in user.model_dump()

try:
    User.model_validate({**user_data, "email": "invalid-email"})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])
else:
    raise AssertionError("Invalid email was accepted")

Modifica el rol a `"developer"` y localiza el error. Después omite el nombre
y compara los mensajes de campo inválido y campo requerido.

## 7. Modelos anidados y serialización

Un reporte contiene configuración, predicciones y errores: cada componente tiene
su propio modelo. `Field(default_factory=list)` crea una lista por instancia.
Una lista de modelos conserva estructura; no necesitamos diccionarios sin contrato
para pasar resultados entre funciones.

Serializar transforma la representación: `model_dump()` produce un diccionario y
`model_dump_json()` texto JSON. `model_validate_json()` reconstruye y valida desde JSON.
No equivale a escribir automáticamente un archivo ni a guardar en una base de datos.

In [ ]:
class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)

In [ ]:
example_report = BatchReport(config=config, accepted=[normalized])
payload = example_report.model_dump()
json_text = example_report.model_dump_json(indent=2)
print(json_text)
assert isinstance(payload["accepted"][0], dict)
restored_report = BatchReport.model_validate_json(json_text)
assert restored_report == example_report

### Ejercicio 6 · Ida y vuelta

Crea un reporte vacío, serialízalo a JSON y reconstrúyelo. Comprueba igualdad
y listas vacías. Después cambia el score a 5 en un diccionario obtenido con
`example_report.model_dump()`: al validarlo debe aparecer la ubicación
`("accepted", 0, "score")`. El modelo original debe conservar score 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Inspect a nested error and check that the source model is unchanged.

## 8. De la notebook a un módulo reutilizable

La siguiente celda reúne las clases anteriores y
las funciones de procesamiento en `lesson_models.py`, el mismo archivo que acompaña
esta notebook en el repositorio. Así Colab funciona sin descargar archivos auxiliares.

**La celda escribe o reemplaza `lesson_models.py` en el directorio de trabajo.**
Si hiciste cambios propios en ese archivo, guárdalos con otro nombre antes de
volver a ejecutarla. `%%writefile` es una instrucción de Jupyter, no sintaxis Python.

Lee primero `process_records`: valida cada entrada, guarda cada problema con su
posición desde cero y no captura excepciones ajenas a validación. Si el lote supera
`max_records`, lanza `ValueError` antes de procesarlo. El umbral selecciona entre
registros válidos; no convierte en inválidas las predicciones con menor score.

Importamos el módulo como `models`: el prefijo distingue sus clases de las
versiones construidas durante la explicación y evita mezclar sus instancias.

In [ ]:
%%writefile lesson_models.py
"""Typed contracts for synthetic prediction records; no model is trained here."""

from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value


class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value


class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)


def average_score(predictions: list[Prediction]) -> float | None:
    if not predictions:
        return None
    return sum(prediction.score for prediction in predictions) / len(predictions)


def process_records(records: list[object], config: BatchConfig) -> BatchReport:
    """Reject an oversized batch; preserve all validation issues for each row."""
    if len(records) > config.max_records:
        raise ValueError("Batch exceeds max_records")

    report = BatchReport(config=config)
    for position, record in enumerate(records):
        try:
            prediction = Prediction.model_validate(record)
        except ValidationError as error:
            for detail in error.errors():
                report.issues.append(
                    ValidationIssue(
                        position=position,
                        field=".".join(str(part) for part in detail["loc"]) or "<record>",
                        message=detail["msg"],
                        error_type=detail["type"],
                    )
                )
        else:
            report.accepted.append(prediction)
    return report


def select_predictions(report: BatchReport) -> list[Prediction]:
    """Select by confidence; this is not an accuracy measurement."""
    return [
        prediction
        for prediction in report.accepted
        if prediction.score >= report.config.threshold
    ]

In [ ]:
import importlib
import lesson_models as models

# Reload after running the file-writing cell again.
models = importlib.reload(models)
from lesson_models import average_score, process_records, select_predictions

stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "lesson_models.py"]
)
print(stdout)
assert exit_status == 0, stderr or stdout

## 9. Práctica guiada: de entradas crudas a reporte

Estos seis registros fueron escritos para esta lección y se pueden reutilizar con
los materiales del curso. Incluyen dos válidos, un score fuera de rango, una
etiqueta nula, un score ausente y una entrada que no es un diccionario.

Antes de ejecutar: identifica posiciones válidas y rechazadas. Esperamos **2
aceptadas**, **4 posiciones rechazadas**, promedio **0.49** y **1 seleccionada** con
umbral 0.9. No calculamos exactitud: no existen etiquetas de referencia.

In [ ]:
raw_records: list[object] = [
    {"text": " Excellent ", "label": " POSITIVE ", "score": "0.98"},
    {"text": "Zero is valid", "label": "neutral", "score": 0.0},
    {"text": "Out of range", "label": "negative", "score": 1.2},
    {"text": "Missing label", "label": None, "score": 0.8},
    {"text": "Missing score", "label": "neutral"},
    None,
]

batch_config = models.BatchConfig(batch_name="Class demo", threshold=0.9, max_records=10)
report = process_records(raw_records, batch_config)
rejected_positions = {issue.position for issue in report.issues}
print(f"Accepted: {len(report.accepted)}")
print(f"Rejected records: {len(rejected_positions)}")
print(f"Average score: {average_score(report.accepted)}")
print(f"Selected: {len(select_predictions(report))}")
for issue in report.issues:
    print(issue.position, issue.field, issue.error_type)

### Ejercicio 7 · Comprobar antes de ampliar

Verifica los resultados esperados del lote. Procesa una lista vacía y otra con
solo `None`: sus promedios deben ser `None`. Prueba un lote mayor que el máximo:
debe producir `ValueError`, sin truncarlo silenciosamente. Usa el módulo de apoyo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Write assertions for the demo batch and the boundary cases.

### Ejercicio 8 · Actividad independiente

Escribe una función tipada `count_labels(predictions)` que devuelva un diccionario
de conteos por etiqueta. Usa `list[models.Prediction]` como entrada y `dict[str, int]`
como salida. Debe funcionar para lista vacía. Añádela a una copia de tu módulo,
compruébala y ejecuta mypy sobre esa copia. Dentro de ese módulo usa
`list[Prediction]`, porque la clase se define allí; el prefijo `models` se usa
solo desde esta notebook.

Como parte de la misma evidencia, crea un registro con **dos errores**, conserva
ambos y explica por qué cuenta como **un registro rechazado**. Serializa el reporte
y comprueba que se pueda reconstruir.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [ ]:
# Implement count_labels and check it with validated predictions.
# Add the function to your own module copy after trying it here.

## Evidencia y criterios

Conserva tu módulo con la función del ejercicio 8,
las comprobaciones y un reporte JSON. Puedes copiar la salida JSON o guardarla
localmente; todavía no se exige una base de datos ni un proyecto empaquetado.

- **Contratos claros:** parámetros y retornos anotados; ausencia de `Any` injustificado.
- **Validación:** restricciones y normalización justificadas; cero válido, faltantes y
  entradas inválidas distinguidos; configuración validada antes de procesar.
- **Errores útiles:** posición, campo y motivo conservados, incluidos varios errores por registro.
- **Serialización:** reconstrucción equivalente del reporte anidado.
- **Verificación:** mypy sin errores en el módulo final y comprobaciones de casos límite.

### Profundización

Compara `model_dump(exclude_none=True)` con la salida habitual. ¿Cuál es la diferencia
entre omitir `source` en la representación y cambiar el contrato del modelo?

## Cierre

Explica por qué una anotación no reemplaza la validación, qué información aporta
un error de Pydantic y qué decisiones de conversión tomaste. La próxima sesión
procesa datos de forma perezosa con iteradores, generadores y context managers.

## Referencias

- [Pydantic: modelos](https://docs.pydantic.dev/latest/concepts/models/).
- [Pydantic: validadores](https://docs.pydantic.dev/latest/concepts/validators/).
- [mypy: primeros pasos](https://mypy.readthedocs.io/en/stable/getting_started.html).

- [PEP 1: propósito y proceso](https://peps.python.org/pep-0001/).
- [PEP 484: Type Hints](https://peps.python.org/pep-0484/).
